# Simplified Copy-Trade Backtest

Simulates copying trades of a hand-picked set of wallets against the
processed Polygon trade shards.

## Strategy
- When a watched wallet prints a **BUY** on token `(condition_id, outcome)` at price `p`
  we place a buy order at a **slightly worse** price: `order_price = clip(p + WORSE_PRICE_OFFSET, 0.001, 0.999)`.
- The order is matched against the **first subsequent trade** whose effective price is
  `<= order_price` within `FILL_HORIZON_SECONDS`.
  - *same-token* liquidity: later BUY trades on the same `(condition_id, outcome)`.
  - *opposite-token* liquidity: later SELL trades on the **complementary** outcome
    (binary markets only), with effective price = `1 - sell_price`.
- For a wallet **SELL** the mirror applies: order at `p - WORSE_PRICE_OFFSET`, filled
  by the first later trade with effective price `>= order_price`.
  - same-token: later SELL prints on the same outcome.
  - opposite-token: later BUY prints on the complementary outcome, effective price `1 - buy_price`.

## Sharding
Trades are partitioned by the first hex character of `condition_id` (after `0x`).
All shards are processed in parallel; within each shard the backtest is exact
because a `condition_id` always falls in exactly one shard file.

## Outputs
One event ledger DataFrame with columns:
- `row_type` — `trigger` (watched-wallet trade that generated an order) or `fill` (our fill).
- `order_id` — unique integer identifying the copy order.
- `wallet` — originating wallet (`fill` rows carry the trigger wallet for reference).
- `condition_id`, `outcome`, `side`, `dt`, `tx_hash`, `price`.
- `order_price` — the price our order was placed at (`NaN` for fill rows).
- `fill_source` — `same_token` / `opposite_token` / `NaN` for trigger rows.
- `token_winner` — market resolution flag.
- `fill_pnl_usdc` — PnL of *our* position on fill rows (NaN for trigger rows),
  computed as stake × ( 1/exec_price − 1 ) for winning tokens, −stake for losers.
- `filled_wallet_total_position` — total filled position for this wallet across all conditions/outcomes.
- `filled_token_total_position` — total filled position for this `(condition_id, outcome)` across all wallets.
- `filled_token_by_wallet_position` — filled position for this `(wallet, condition_id, outcome)`.

## Visualisation
Cumulative PnL chart with two series:
- **Wallet cohort** — raw `trade_pnl` of the watched wallets.
- **Copy strategy** — `fill_pnl_usdc` of our simulated fills.


## Configuration

In [ ]:
%load_ext autoreload
%autoreload 2

import datetime
from pathlib import Path

# ── Input wallets ─────────────────────────────────────────────────────────────
# Replace with any wallet addresses you want to copy.
# Example: load from a CSV, a workspace registry, or define inline.
WATCHED_WALLETS: list[str] = None

ONLY_BUY_TRIGGERS = False
MAX_EXPOSURE_PER_WALLET_1h = 200

# ── Test period ───────────────────────────────────────────────────────────────
# None = use entire dataset (start/end are inferred from the data).
# Set to datetime.date objects to restrict the window.
PERIOD_START: datetime.date | None = datetime.date(2026, 4, 16)
# PERIOD_START: datetime.date | None = datetime.date(2026, 1, 1) # None   # e.g. datetime.date(2026, 2, 16)
PERIOD_END:   datetime.date | None = datetime.date(2026, 6, 20)
# PERIOD_END:   datetime.date | None = datetime.date(2026, 2, 16) # None   # e.g. datetime.date(2026, 3, 11)

# ── Copy-trade execution parameters ──────────────────────────────────────────
WORSE_PRICE_OFFSET: float = 0
FILL_HORIZON_SECONDS: float = 300     # max seconds after trigger to search for a fill
STAKE_USDC: float = 10.0               # max USDC per order (order qty = min(trigger_qty, STAKE_USDC / order_price))
FEE_BPS: float = 10.0                   # taker fee in basis points
MAX_POSITION_PER_CONDITION: float | None = 1000  # max qty per (wallet, condition_id) across all outcomes; None = unlimited
MAX_POSITION_PER_WALLET:    float | None = 100  # max qty per wallet across all conditions; None = unlimited

# ── Data ─────────────────────────────────────────────────────────────────────
TRADES_DIR     = Path("../../data/polygon_trades_processed")
RAW_TRADES_DIR = Path("../../data/trades_polygon")

# ── Parallelism ───────────────────────────────────────────────────────────────
MAX_WORKERS: int = 10   # number of threads for parallel shard processing

WORKSPACE_DIR = Path("../../data/trade_signals_workspace_v2")
wallets_path = WORKSPACE_DIR / "selected_wallets_v2.parquet"

In [80]:
CONDITION='0xfbeac45ae53828674baf51e12313b7fb651594915cfb082e7cc8694896f6fc21'


# df = pd.read_parquet(TRADES_DIR / (CONDITION[2] + ".parquet"))
df = pd.concat(pd.read_parquet(p) for p in TRADES_DIR.glob("*.parquet"))
df = df[df['is_train'] == False].copy()

In [82]:
print(len(df[df['condition_id'] == CONDITION]))
len(df)

191


3905163

In [73]:
wdf = pd.read_parquet(wallets_path)

In [63]:
wdf.head()

,wallet,pnl_volatility,num_buckets,num_markets,total_notional,total_pnl,copyable_pnl,top5_pnl_pct,top10_pnl_pct,worst5_pnl_pct,...,sample_mult,downside_tail,predictability_score,predicting_final_score,copyable_efficiency,copyable_dd_ratio,copyable_score,final_score,wallet_quality,wallet_group
0,0x3e438fdb437d90925b47b6f8ab85405ba8ffa83f,0.988890,56,34,5978.575157,1037.159066,644.303983,0.538294,0.787085,-0.269878,...,0.531917,0.269878,-0.582112,-0.582112,0.107751,0.114708,4.314947,1.376712,1.376712,copyable
1,0xb78ddd792ad7d1f603fcd046c662d6d9adbf835d,1.520942,124,67,13154.695188,2148.207585,1169.647364,0.433823,0.675410,-0.342110,...,0.635229,0.342110,-1.192221,-1.192221,0.088908,0.091842,2.915341,0.450804,0.450804,copyable
2,0xe97b7b1396a99341b32984fa8bbf62f2f68ab2ea,1.362514,221,47,50819.144170,6107.897999,1358.211608,0.396656,0.622696,-0.212643,...,0.710794,0.212643,-0.996668,-0.996668,0.026726,0.330738,1.038844,-0.182463,-0.182463,copyable
3,0x2853240a0f4e9e11a949a5cfa6e0fe953a293482,0.916907,3812,1106,225426.076885,29006.484985,4236.017165,0.204374,0.321428,-0.048112,...,1.000000,0.048112,-0.901940,-0.901940,0.018791,0.131244,0.626686,-0.290490,-0.290490,copyable
4,0x0cb10c40b0776e9ee8cef970af85724654dda76c,1.035172,5472,1784,299213.399270,46376.749251,6731.762398,0.139357,0.215746,-0.058581,...,1.000000,0.058581,-1.103916,-1.103916,0.022498,0.160811,0.825330,-0.332217,-0.332217,copyable


In [64]:
copyable_set = set(wdf[wdf['wallet_group'] == 'copyable']['wallet'].to_list())
watched_set = set(wdf['wallet'].to_list())

In [74]:
df[df['condition_id'] == '0xf25804c3fe472903e37da74c0e80520c397e6be15a55013d3c7d39117adeb783'].sort_values('dt', ascending=False).head(20)

,wallet,condition_id,dt,side,outcome,position,total_quantity,avg_price,trade_value_usdc,final_value_usdc,trade_pnl,copyable_pnl,token_winner,final_price,tx_hash,num_fills,is_train,copyable_qty,avail_copy_total_vol,avail_copy_count


In [96]:
df['is_copyable_wallet'] = df['wallet'].isin(copyable_set)
print(f"Copyable wallets: {len(copyable_set)}, copyable fills: {df['is_copyable_wallet'].sum()} total pnl: {df[df['is_copyable_wallet']]['trade_pnl'].sum():,.2f}")
df['is_watched_wallet'] = df['wallet'].isin(watched_set)
print(f"Watched wallets: {len(watched_set)}, watched fills: {df['is_watched_wallet'].sum()} total pnl: {df[df['is_watched_wallet']]['trade_pnl'].sum():,.2f}")

Copyable wallets: 32, copyable fills: 11357 total pnl: 53,127.72
Watched wallets: 122, watched fills: 842264 total pnl: 231,607.16


In [ ]:
df[df['is_copyable_wallet'] == True]

11357

In [92]:
(
    df.assign(
        watched_wallet_pnl=df["trade_pnl"] * df["is_watched_wallet"],
        copyable_wallet_pnl=df["trade_pnl"] * df["is_copyable_wallet"]
    )
    .groupby("condition_id")
    .agg(
        fills_count=("condition_id", "count"),
        wallet_set_fills_count=("is_watched_wallet", "sum"),
        copyable_wallet_set_fills_count=("is_copyable_wallet", "sum"),
        watched_wallets_pnl=("watched_wallet_pnl", "sum"),
        copyable_wallets_pnl=("copyable_wallet_pnl", "sum"),
    )
    .sort_values("wallet_set_fills_count", ascending=False)
    .iloc[0:]
    .head(20)
).sum()

fills_count                        359900.000000
wallet_set_fills_count              36694.000000
copyable_wallet_set_fills_count      2865.000000
watched_wallets_pnl                 -2360.287042
copyable_wallets_pnl               -16082.139298
dtype: float64

In [218]:
df[df['wallet'].isin(wallet_set)].groupby("condition_id")['condition_id'].count().sort_values(ascending=False).head(20)

condition_id
0xfb5f415a77fb16d4a96d19909c86f93204b8835664f3bfa9af92a349e2d30a25    4187
0xf6890c707d1e327b006b9511bdba8c98ec4e4978646c6ffcf3ec8750c26a666a    2784
0xf32e257000f1a5c494fb3e0ad7bc171bd23d95c734668a5fb9e8a03e1570820d    2353
0xf9f0965a3bd704ad9c184851242606ad35a4ffe733febf5074d4fbcc59d52eb8    2189
0xf005ddced69842c3943edea3b8fb24187240ac93becdc1532c5b9db34f9dd2c8    1343
0xfe4ddfdf4e6b6188967331f5932c8ec6dcbc815572ef9f3566b5d903a708b61f    1127
0xfc3fd061267a264175e222ed643f6b9d958a2627ad6c631290838de3895c118f    1123
0xfa1543cdef36d55ef9126aaab6015c7c7ed5aa6a2bb5be355f5cacc2302c7374    1076
0xf335cc3103bdc40edc68be074c7a9debc9d9c47845024f728cb76875b29e0a26     967
0xfaeb9d120a69ad3bdc3b7f6d3b0965a1960a998a4469241dd796c2c96b232832     779
0xf526b2fbff94ae996818e76c8b54cc99ffbfa0a1a9972a48253ada65e32786f7     705
0xf82ce6cd43d102b5545be8c107dda25697417a7af42ae277c5c8de0f812ae4fb     702
0xffb8a1a5c71eacf9b8dbe4f10b846c239c74680756cb744fcfc760db45eb256e     696
0xf8b7308102

In [199]:
df.groupby("condition_id")['condition_id'].count().sort_values(ascending=False).head(20)

condition_id
0xf2ce8d3897ac5009a131637d3575f1f91c579bd08eecce6ae2b2da0f32bbe6f1    57356
0xf6890c707d1e327b006b9511bdba8c98ec4e4978646c6ffcf3ec8750c26a666a    50933
0xf64f880b571d7a70d858649d30f0843aa57307e304aeb617349df74ce34d044e    39032
0xf9f0965a3bd704ad9c184851242606ad35a4ffe733febf5074d4fbcc59d52eb8    26020
0xfe4ddfdf4e6b6188967331f5932c8ec6dcbc815572ef9f3566b5d903a708b61f    24245
0xfa48a99317daef1654d5b03e30557c4222f276657275628d9475e141c64b545d    20634
0xfc9ee9c25abd5b966088767214c22dcf3ad1c72548e9a1dbef9d4328c170e5b5    20584
0xf963ec4b153e66668dd9e038a2cb258a3d228fde87efdcde95e1e3e8fc6ec56a    18515
0xf005ddced69842c3943edea3b8fb24187240ac93becdc1532c5b9db34f9dd2c8    18326
0xf6e44da63acf52f67bd2d370468c3ef297f064c50f660370151d6c8f08fd586d    14282
0xfb5f415a77fb16d4a96d19909c86f93204b8835664f3bfa9af92a349e2d30a25    12170
0xf94145fa5716d4bfd2948ce5eef2954dcb560516ccbba5e8b68059027e431cdc    11732
0xf7ebe673ed7e71d3846b4f0c74065351d9f7f2717088ea2a517d86ef0d02c7a8    11034

In [56]:
len(WATCHED_WALLETS)

122

In [12]:
banned_wallets = set()
if(WATCHED_WALLETS is not None):
    WATCHED_WALLETS = [w for w in WATCHED_WALLETS if w not in banned_wallets]

## Optionally load wallets from stage-1 workspace

If `WATCHED_WALLETS` is empty above, this cell loads the wallet set from the
stage-1 strategy registry. Skip or modify as needed.

In [ ]:
import pandas as pd

if not WATCHED_WALLETS:
    if wallets_path.exists():
        _wallets_df = pd.read_parquet(wallets_path, columns=["wallet"])
        WATCHED_WALLETS = _wallets_df["wallet"].tolist()
        print(f"Loaded {len(WATCHED_WALLETS)} wallets from {wallets_path.name}")
    else:
        print("No wallet source found — set WATCHED_WALLETS manually or run stage1 first.")
else:
    print(f"Using {len(WATCHED_WALLETS)} manually specified wallets.")

Using 122 manually specified wallets.


## Discover shards and derive test period

In [14]:
import pandas as pd

SHARD_PATHS: list[Path] = sorted(TRADES_DIR.glob("*.parquet"))
print(f"Found {len(SHARD_PATHS)} shards under {TRADES_DIR}")

# Derive END_DATE_TRAIN from the is_train flag (last date where is_train=True).
# Test data is everything strictly after END_DATE_TRAIN.
_sample = pd.read_parquet(SHARD_PATHS[0], columns=["dt", "is_train"])
_sample["dt"] = pd.to_datetime(_sample["dt"], utc=True)
END_DATE_TRAIN: datetime.date = _sample.loc[_sample["is_train"], "dt"].max().date()
_data_end: datetime.date      = _sample["dt"].max().date()
del _sample
print(f"END_DATE_TRAIN (last train date) : {END_DATE_TRAIN}")

# Backtest always runs on test data only (strictly after END_DATE_TRAIN).
# PERIOD_START / PERIOD_END can narrow the window further, but cannot go
# earlier than the day after END_DATE_TRAIN.
#_test_start = END_DATE_TRAIN + datetime.timedelta(days=1)
period_start: datetime.date = PERIOD_START # datetime.date(2026, 2, 16) #  max(PERIOD_START, _test_start) if PERIOD_START is not None else _test_start
period_end:   datetime.date = PERIOD_END #if PERIOD_END is not None else _data_end
print(f"Backtest period : {period_start}  →  {period_end}")

Found 16 shards under ../../data/polygon_trades_processed
END_DATE_TRAIN (last train date) : 2026-04-15
Backtest period : 2026-04-16  →  2026-06-20


## Backtest engine (per-shard)

Each shard is processed independently:
1. Load only rows within the backtest period (test data only).
2. Identify trigger events (watched-wallet trades).
3. Build a per-`condition_id` opposite-outcome map (binary markets only).
4. Process triggers in time order, maintaining a copy-portfolio **position** per `(wallet, condition_id, outcome)`.
5. For each trigger, place a copy order:
   - **BUY**: `qty_ordered = min(trig_qty, STAKE_USDC / order_price)` — worst-case loss = `qty × order_price ≤ STAKE_USDC`
   - **SELL**: `qty_ordered = min(trig_qty, position, STAKE_USDC / (1 − order_price))` — worst-case loss = `qty × (1 − order_price) ≤ STAKE_USDC`. Skipped if position = 0 (no short selling).
6. Scan tape prints in time order within `FILL_HORIZON_SECONDS`. Each matching print partially fills the order: `fill_qty = min(remaining_qty, tape_qty)`.
7. One **fill** ledger row is emitted per partial fill. BUY fills increment position; SELL fills decrement it.

`fill_pnl_usdc` per fill row (all executed at `exec_price = order_price`, limit fill):
- **BUY winner**: `fill_qty × (1 − exec_price) − fill_qty × exec_price × fee`
- **BUY loser**:  `−fill_qty × exec_price × (1 + fee)`
- **SELL winner** (sold below par): `fill_qty × (exec_price − 1) − fill_qty × exec_price × fee`
- **SELL loser** (sold above par):  `fill_qty × exec_price × (1 − fee)`


In [15]:
import numpy as np
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed

# ─────────────────────────────────────────────────────────────────────────────
# Helpers
# ─────────────────────────────────────────────────────────────────────────────

def _build_complement_map(df: pd.DataFrame) -> dict[tuple[str, str], str]:
    """Return {(condition_id, outcome): opposite_outcome} for binary markets."""
    pairs: dict[str, set] = {}
    for cid, grp in df.groupby("condition_id", sort=False):
        pairs[cid] = set(grp["outcome"].dropna().unique())
    result: dict[tuple[str, str], str] = {}
    for cid, outcomes in pairs.items():
        if len(outcomes) == 2:
            a, b = sorted(outcomes)
            result[(cid, a)] = b
            result[(cid, b)] = a
    return result


def _simulate_shard(
    shard_path: Path,
    raw_tape_path: Path,
    watched_wallets: set[str],
    period_start: datetime.date,
    period_end: datetime.date,
    worse_price_offset: float,
    fill_horizon_seconds: float,
    stake_usdc: float,
    fee_bps: float,
    max_position_per_condition: float | None = None,
    max_position_per_wallet: float | None = None,
) -> pd.DataFrame:
    """Process one shard file and return an event ledger DataFrame.

    Triggers are read from ``shard_path`` (processed per-wallet aggregated trades),
    which supplies ``token_winner`` and ``trade_pnl``.

    The fill tape is read from ``raw_tape_path`` (raw on-chain individual fills).
    Orders are simulated chronologically against the tape so that each tape print's
    quantity can only be consumed once across all active copy orders in the shard.
    """
    TRIG_COLS = [
        "wallet", "condition_id", "outcome", "dt", "side",
        "avg_price", "total_quantity", "trade_pnl", "token_winner",
        "tx_hash",
    ]
    tdf = pd.read_parquet(shard_path, columns=TRIG_COLS)
    if tdf.empty:
        return pd.DataFrame()

    tdf["dt"] = pd.to_datetime(tdf["dt"], utc=True)
    tdf = tdf.rename(columns={"avg_price": "price", "total_quantity": "quantity"})

    date_mask = (
        (tdf["dt"].dt.date >= period_start)
        & (tdf["dt"].dt.date <= period_end)
    )
    tdf = tdf[date_mask].copy()
    if tdf.empty:
        return pd.DataFrame()

    tdf["price"] = tdf["price"].astype(float)
    tdf["quantity"] = tdf["quantity"].astype(float)

    if ONLY_BUY_TRIGGERS:
        trigger_mask = (tdf["wallet"].isin(watched_wallets)) & (tdf["side"] == "BUY")
    else:
        trigger_mask = tdf["wallet"].isin(watched_wallets)
    triggers_df = tdf[trigger_mask].copy()
    if triggers_df.empty:
        return pd.DataFrame()

    tw_map: dict[tuple[str, str], bool] = {}
    for row in tdf[["condition_id", "outcome", "token_winner"]].itertuples(index=False):
        key = (row.condition_id, row.outcome)
        if key not in tw_map and row.token_winner is not None:
            tw_map[key] = bool(row.token_winner)

    complement = _build_complement_map(tdf)
    triggers_df = triggers_df.sort_values("dt", kind="mergesort").reset_index(drop=True)

    TAPE_COLS = ["condition_id", "outcome", "block_timestamp", "side", "price", "quantity", "tx_hash", "token_winner"]
    if raw_tape_path.exists():
        rdf = pd.read_parquet(raw_tape_path, columns=TAPE_COLS)
    else:
        rdf = pd.DataFrame(columns=TAPE_COLS)

    if rdf.empty:
        ledger_rows = []
        for trig in triggers_df.itertuples(index=False):
            cid = trig.condition_id
            outcome = trig.outcome
            side = trig.side
            trig_dt = trig.dt
            trig_px = float(trig.price)
            trig_qty = float(trig.quantity)
            trig_tw = bool(trig.token_winner)
            wallet = trig.wallet
            if side == "BUY":
                order_px = float(np.clip(trig_px + worse_price_offset, 0.001, 0.999))
                qty_ordered = min(trig_qty, stake_usdc / order_px)
            else:
                order_px = float(np.clip(trig_px - worse_price_offset, 0.001, 0.999))
                qty_ordered = trig_qty
            ledger_rows.append({
                "row_type": "trigger", "order_id": len(ledger_rows) + 1,
                "wallet": wallet, "condition_id": cid, "outcome": outcome,
                "side": side, "dt": trig_dt, "tx_hash": trig.tx_hash,
                "price": trig_px, "order_price": order_px,
                "qty_ordered": qty_ordered, "qty_filled": 0.0,
                "fill_qty": None, "fill_source": None,
                "token_winner": trig_tw, "exec_price": None,
                "fill_pnl_usdc": None,
                "wallet_trade_pnl": float(trig.trade_pnl) if trig.trade_pnl is not None else None,
                "filled_wallet_total_position": 0.0,
                "filled_token_total_position": 0.0,
                "filled_token_by_wallet_position": 0.0,
            })
        result = pd.DataFrame(ledger_rows)
        result["shard"] = shard_path.stem
        return result

    rdf["dt"] = pd.to_datetime(rdf["block_timestamp"], unit="s", utc=True)
    rdf = rdf.drop(columns=["block_timestamp"])

    tape_start = pd.Timestamp(period_start, tz="UTC")
    tape_end = pd.Timestamp(period_end, tz="UTC") + pd.Timedelta(days=1)
    rdf = rdf[(rdf["dt"] >= tape_start) & (rdf["dt"] < tape_end)].copy()
    rdf["price"] = rdf["price"].astype(float)
    rdf["quantity"] = rdf["quantity"].astype(float)
    rdf = rdf.sort_values("dt", kind="mergesort").reset_index(drop=True)

    fee = fee_bps / 10_000.0
    horizon = pd.Timedelta(seconds=fill_horizon_seconds)
    eps = 1e-12

    ledger_rows: list[dict] = []
    order_counter = 0
    # (wallet, condition_id, outcome) → filled qty for this outcome
    positions: dict[tuple[str, str, str], float] = defaultdict(float)
    # (wallet, condition_id) → total filled qty across all outcomes of that condition
    cond_positions: dict[tuple[str, str], float] = defaultdict(float)
    # (condition_id, outcome) → total filled qty for this outcome across all wallets
    token_total_positions: dict[tuple[str, str], float] = defaultdict(float)
    # wallet → total filled qty across all conditions
    wallet_positions: dict[str, float] = defaultdict(float)

    orders: dict[int, dict] = {}
    books: dict[tuple[str, str, str], list[dict]] = defaultdict(list)

    def _append_trigger_row(
        order_id: int,
        trig,
        order_px: float,
        qty_ordered: float,
        trig_tw: bool,
        current_token_pos: float = 0.0,
        current_token_total_pos: float = 0.0,
        current_wallet_pos: float = 0.0,
    ) -> None:
        ledger_rows.append({
            "row_type": "trigger",
            "order_id": order_id,
            "wallet": trig.wallet,
            "condition_id": trig.condition_id,
            "outcome": trig.outcome,
            "side": trig.side,
            "dt": trig.dt,
            "tx_hash": trig.tx_hash,
            "price": float(trig.price),
            "order_price": order_px,
            "qty_ordered": qty_ordered,
            "qty_filled": None,
            "fill_qty": None,
            "fill_source": None,
            "token_winner": trig_tw,
            "exec_price": None,
            "fill_pnl_usdc": None,
            "wallet_trade_pnl": float(trig.trade_pnl) if trig.trade_pnl is not None else None,
            "filled_wallet_total_position": current_wallet_pos,
            "filled_token_total_position": current_token_total_pos,
            "filled_token_by_wallet_position": current_token_pos,
        })

    # order can match with trade on the same side and token, or opposite side and opposite token    
    def _register_order(order_id: int, order: dict, trig_tw: bool) -> None:
        cid = order["condition_id"]
        outcome = order["outcome"]
        side = order["side"]
        books[(cid, outcome, side)].append({
            "order_id": order_id,
            "fill_source": "same_token",
            "fill_tw": trig_tw,
            "opposite": False,
        })
        opp_outcome = complement.get((cid, outcome))
        if opp_outcome is None:
            return
        opp_tw = tw_map.get((cid, opp_outcome), not trig_tw)
        opp_tape_side = "SELL" if side == "BUY" else "BUY"
        books[(cid, opp_outcome, opp_tape_side)].append({
            "order_id": order_id,
            "fill_source": "opposite_token",
            "fill_tw": trig_tw,
            "opposite": True,
        })

    def _process_tape_row(row) -> None:
        key = (row.condition_id, row.outcome, row.side)
        entries = books.get(key)
        if not entries:
            return

        tape_dt = row.dt
        tape_price = float(row.price)
        tape_remaining = float(row.quantity)
        if tape_remaining <= eps:
            return

        survivors: list[dict] = []
        for entry in entries:
            order = orders.get(entry["order_id"])
            if order is None:
                continue
            if order["remaining_qty"] <= eps:
                continue
            if order["deadline"] < tape_dt:
                continue

            eff_price = float(np.clip(1.0 - tape_price, 0.001, 0.999)) if entry["opposite"] else tape_price
            eligible = eff_price <= order["order_price"] if order["side"] == "BUY" else eff_price >= order["order_price"]

            if eligible and tape_remaining > eps:
                fill_qty = min(order["remaining_qty"], tape_remaining)

                # Cap fill to remaining position headroom (per-condition and per-wallet limits).
                # This prevents overfill when a single tape print is large enough to exceed the cap.
                if order["side"] == "BUY":
                    if max_position_per_condition is not None:
                        cond_headroom = max(max_position_per_condition - cond_positions[(order["wallet"], order["condition_id"])], 0.0)
                        fill_qty = min(fill_qty, cond_headroom)
                    if max_position_per_wallet is not None:
                        wallet_headroom = max(max_position_per_wallet - wallet_positions[order["wallet"]], 0.0)
                        fill_qty = min(fill_qty, wallet_headroom)
                    if fill_qty <= eps:
                        # Position cap already reached — retire this order without consuming tape
                        order["remaining_qty"] = 0.0
                        continue

                tape_remaining -= fill_qty
                order["remaining_qty"] -= fill_qty

                if order["side"] == "BUY":
                    gross = fill_qty * (1.0 - order["order_price"]) if entry["fill_tw"] else -fill_qty * order["order_price"]
                    fill_pnl = gross - fill_qty * order["order_price"] * fee
                    positions[order["pos_key"]] += fill_qty
                    cond_positions[(order["wallet"], order["condition_id"])] += fill_qty
                    token_total_positions[(order["condition_id"], order["outcome"])] += fill_qty
                    wallet_positions[order["wallet"]] += fill_qty
                    new_token_pos = positions[order["pos_key"]]
                    new_token_total_pos = token_total_positions[(order["condition_id"], order["outcome"])]
                    new_wallet_pos = wallet_positions[order["wallet"]]
                else:
                    gross = fill_qty * (order["order_price"] - 1.0) if entry["fill_tw"] else fill_qty * order["order_price"]
                    fill_pnl = gross - fill_qty * order["order_price"] * fee
                    prev = positions[order["pos_key"]]
                    positions[order["pos_key"]] = max(prev - fill_qty, 0.0)
                    actually_reduced = prev - positions[order["pos_key"]]
                    cond_positions[(order["wallet"], order["condition_id"])] = max(
                        cond_positions[(order["wallet"], order["condition_id"])] - actually_reduced, 0.0
                    )
                    wallet_positions[order["wallet"]] = max(
                        wallet_positions[order["wallet"]] - actually_reduced, 0.0
                    )
                    token_total_positions[(order["condition_id"], order["outcome"])] = max(
                        token_total_positions[(order["condition_id"], order["outcome"])] - actually_reduced, 0.0
                    )
                    new_token_pos = positions[order["pos_key"]]
                    new_token_total_pos = token_total_positions[(order["condition_id"], order["outcome"])]
                    new_wallet_pos = wallet_positions[order["wallet"]]

                ledger_rows.append({
                    "row_type": "fill",
                    "order_id": entry["order_id"],
                    "wallet": order["wallet"],
                    "condition_id": order["condition_id"],
                    "outcome": order["outcome"],
                    "side": order["side"],
                    "dt": tape_dt,
                    "tx_hash": row.tx_hash,
                    "price": eff_price,
                    "order_price": None,
                    "qty_ordered": order["qty_ordered"],
                    "qty_filled": None,
                    "fill_qty": fill_qty,
                    "fill_source": entry["fill_source"],
                    "token_winner": entry["fill_tw"],
                    "exec_price": order["order_price"],
                    "fill_pnl_usdc": fill_pnl,
                    "wallet_trade_pnl": None,
                    "filled_wallet_total_position": new_wallet_pos,
                    "filled_token_total_position": new_token_total_pos,
                    "filled_token_by_wallet_position": new_token_pos,
                })

            if order["remaining_qty"] > eps and order["deadline"] >= tape_dt:
                survivors.append(entry)

        if survivors:
            books[key] = survivors
        else:
            books.pop(key, None)

    tape_iter = iter(rdf[["condition_id", "outcome", "dt", "side", "price", "quantity", "tx_hash", "token_winner"]].itertuples(index=False))
    current_tape = next(tape_iter, None)

    for trig in triggers_df.itertuples(index=False):
        while current_tape is not None and current_tape.dt <= trig.dt:
            _process_tape_row(current_tape)
            current_tape = next(tape_iter, None)

        cid = trig.condition_id
        outcome = trig.outcome
        side = trig.side
        trig_px = float(trig.price)
        trig_qty = float(trig.quantity)
        trig_tw = bool(trig.token_winner)
        wallet = trig.wallet
        pos_key = (wallet, cid, outcome)

        if side == "BUY":
            order_px = float(np.clip(trig_px + worse_price_offset, 0.001, 0.999))
            current_pos = positions.get(pos_key, 0.0)
            qty_ordered = min(trig_qty, stake_usdc / order_px)
        else:
            order_px = float(np.clip(trig_px - worse_price_offset, 0.001, 0.999))
            current_pos = positions.get(pos_key, 0.0)
            if current_pos <= eps:
                continue
            sell_cap = stake_usdc / max(1.0 - order_px, 1e-9)
            qty_ordered = min(trig_qty, current_pos, sell_cap)
            if qty_ordered <= eps:
                continue

        order_counter += 1
        order_id = order_counter

        order = {
            "wallet": wallet,
            "condition_id": cid,
            "outcome": outcome,
            "side": side,
            "order_price": order_px,
            "qty_ordered": qty_ordered,
            "remaining_qty": qty_ordered,
            "deadline": trig.dt + horizon,
            "pos_key": pos_key,
        }
        orders[order_id] = order

        _append_trigger_row(
            order_id,
            trig,
            order_px,
            qty_ordered,
            trig_tw,
            current_token_pos=positions.get(pos_key, 0.0),
            current_token_total_pos=token_total_positions.get((cid, outcome), 0.0),
            current_wallet_pos=wallet_positions.get(wallet, 0.0),
        )
        _register_order(order_id, order, trig_tw)

    while current_tape is not None:
        _process_tape_row(current_tape)
        current_tape = next(tape_iter, None)

    if not ledger_rows:
        return pd.DataFrame()

    result = pd.DataFrame(ledger_rows)
    result["shard"] = shard_path.stem

    filled_by_order = (
        result[result["row_type"] == "fill"]
        .groupby("order_id")["fill_qty"]
        .sum()
        .rename("_total_filled")
    )
    result = result.merge(filled_by_order, on="order_id", how="left")
    trig_mask = result["row_type"] == "trigger"
    result.loc[trig_mask, "qty_filled"] = result.loc[trig_mask, "_total_filled"].fillna(0.0)
    result = result.drop(columns=["_total_filled"])
    result["qty_filled"] = result["qty_filled"].astype(float)

    return result



## Run backtest across all shards (parallel)

In [16]:
assert WATCHED_WALLETS, "WATCHED_WALLETS is empty — set wallets in the config cell or run the wallet-load cell."

watched_set = set(WATCHED_WALLETS)
shard_results: list[pd.DataFrame] = []

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {
        executor.submit(
            _simulate_shard,
            path,
            RAW_TRADES_DIR / path.name,
            watched_set,
            period_start,
            period_end,
            WORSE_PRICE_OFFSET,
            FILL_HORIZON_SECONDS,
            STAKE_USDC,
            FEE_BPS,
            MAX_POSITION_PER_CONDITION,
            MAX_POSITION_PER_WALLET,
        ): path
        for path in SHARD_PATHS
    }
    for future in as_completed(futures):
        path = futures[future]
        try:
            df = future.result()
            if df is not None and not df.empty:
                shard_results.append(df)
        except Exception as exc:
            print(f"Shard {path.name} raised an exception: {exc}")
            raise

if shard_results:
    event_ledger: pd.DataFrame = (
        pd.concat(shard_results, ignore_index=True)
        .sort_values(["dt", "shard", "order_id", "row_type"])
        .reset_index(drop=True)
    )
    _key = event_ledger[["shard", "order_id"]]
    _pairs = _key.drop_duplicates().reset_index(drop=True)
    _pairs["global_order_id"] = _pairs.index + 1
    event_ledger = event_ledger.merge(_pairs, on=["shard", "order_id"], how="left")
    event_ledger["order_id"] = event_ledger["global_order_id"]
    event_ledger = event_ledger.drop(columns=["global_order_id"])
else:
    event_ledger = pd.DataFrame(columns=[
        "row_type", "order_id", "wallet", "condition_id", "outcome",
        "side", "dt", "tx_hash", "price", "order_price",
        "qty_ordered", "qty_filled", "fill_qty",
        "fill_source", "token_winner", "exec_price", "fill_pnl_usdc",
        "wallet_trade_pnl", "shard", "filled_wallet_total_position", "filled_token_total_position", "filled_token_by_wallet_position",
    ])

triggers = event_ledger[event_ledger["row_type"] == "trigger"]
fills = event_ledger[event_ledger["row_type"] == "fill"]
filled_orders = fills["order_id"].nunique()

print(f"Trigger events    : {len(triggers):>7,}")
print(f"Fill rows         : {len(fills):>7,}")
print(f"Orders with fills : {filled_orders:>7,}")
if len(triggers) > 0:
    print(f"Order fill rate   : {filled_orders/len(triggers)*100:.1f}%")
else:
    print("No trigger events found — check WATCHED_WALLETS and the period.")



Trigger events    : 448,580
Fill rows         :  37,925
Orders with fills :  26,245
Order fill rate   : 5.9%


## Summary statistics

In [17]:
fee = FEE_BPS / 10_000.0

if not fills.empty:
    filled_copy_pnl    = fills["fill_pnl_usdc"].sum()
    total_qty_filled   = fills["fill_qty"].sum()
    total_notional     = (fills["fill_qty"] * fills["exec_price"]).sum()
    orders_with_fills  = fills["order_id"].nunique()
    partial_orders     = (fills.groupby("order_id").size() > 1).sum()
    win_rate           = fills.groupby("order_id")["token_winner"].first().mean()
    avg_exec_price     = fills["exec_price"].mean()
    fill_src_counts    = fills["fill_source"].value_counts()

    # Partial-fill statistics
    trig_qty = triggers.set_index("order_id")["qty_ordered"]
    fill_qty = triggers.set_index("order_id")["qty_filled"]
    fill_pct = (fill_qty / trig_qty.clip(lower=1e-12) * 100).replace([float('inf'), float('nan')], 0)

    print(f"=== Copy-strategy performance ===")
    print(f"  Orders triggered    : {len(triggers):>7,}")
    print(f"  Orders with fills   : {orders_with_fills:>7,}  ({orders_with_fills/len(triggers)*100:.1f}%)")
    print(f"  Orders multi-fill   : {partial_orders:>7,}  ({partial_orders/max(orders_with_fills,1)*100:.1f}% of filled)")
    print(f"  Avg fill %          : {fill_pct[fill_pct>0].mean():>7.1f}%")
    print(f"  Total qty filled    : {total_qty_filled:>10,.2f} tokens")
    print(f"  Total notional      : {total_notional:>10,.2f} USDC")
    print(f"  Net PnL (USDC)      : {filled_copy_pnl:>10,.2f}")
    print(f"  Net ROI on notional : {filled_copy_pnl/total_notional*100:>8.2f}%")
    print(f"  Win rate (by order) : {win_rate*100:>8.2f}%")
    print(f"  Avg exec price      : {avg_exec_price:>8.4f}")
    print(f"\n  Fill source breakdown:")
    for src, cnt in fill_src_counts.items():
        print(f"    {src:<20s}: {cnt:>6,}  ({cnt/len(fills)*100:.1f}%)")
else:
    print("No fills — nothing to summarise.")

# Watched-wallet cohort summary
wallet_pnl = triggers["wallet_trade_pnl"].sum()
print(f"\n=== Watched-wallet cohort ===")
print(f"  Total trades        : {len(triggers):>7,}")
print(f"  Total trade PnL     : {wallet_pnl:>10,.2f} USDC")


=== Copy-strategy performance ===
  Orders triggered    : 448,580
  Orders with fills   :  26,245  (5.9%)
  Orders multi-fill   :   6,990  (26.6% of filled)
  Avg fill %          :    84.4%
  Total qty filled    : 229,263.47 tokens
  Total notional      : 101,813.36 USDC
  Net PnL (USDC)      :     160.15
  Net ROI on notional :     0.16%
  Win rate (by order) :    44.25%
  Avg exec price      :   0.4572

  Fill source breakdown:
    same_token          : 25,713  (67.8%)
    opposite_token      : 12,212  (32.2%)

=== Watched-wallet cohort ===
  Total trades        : 448,580
  Total trade PnL     : 117,724.53 USDC


## Event ledger preview

In [18]:
event_ledger[
    (event_ledger['row_type'] == 'fill')
    & (event_ledger['condition_id'] == '0x3488f31e6449f9803f99a8b5dd232c7ad883637f1c86e6953305a2ef19c77f20')
    & (event_ledger['wallet'] == '0x0b219cf3d297991b58361dbebdbaa91e56b8deb6')
    ].head(500)

,row_type,order_id,wallet,condition_id,outcome,side,dt,tx_hash,price,order_price,...,fill_qty,fill_source,token_winner,exec_price,fill_pnl_usdc,wallet_trade_pnl,filled_wallet_total_position,filled_token_total_position,filled_token_by_wallet_position,shard


In [19]:
pd.set_option('display.max_rows', 100)

In [20]:
display_cols = [
    "row_type", "order_id", "wallet", "condition_id", "outcome", "side",
    "dt", "tx_hash", "price", "order_price", "exec_price",
    "qty_ordered", "qty_filled", "fill_qty",
    "fill_source", "token_winner", "fill_pnl_usdc", "filled_wallet_total_position", "filled_token_total_position", "filled_token_by_wallet_position", "wallet_trade_pnl",
]
available = [c for c in display_cols if c in event_ledger.columns]

# Show a few trigger+fill pairs
sample_orders = event_ledger["order_id"].unique()[:5]
event_ledger[
    event_ledger["order_id"].isin(sample_orders)
][available].sort_values(["order_id", "row_type"])


,row_type,order_id,wallet,condition_id,outcome,side,dt,tx_hash,price,order_price,...,qty_ordered,qty_filled,fill_qty,fill_source,token_winner,fill_pnl_usdc,filled_wallet_total_position,filled_token_total_position,filled_token_by_wallet_position,wallet_trade_pnl
0,trigger,1,0x74cbe13dba27a6a16805e9e7142ee68aa09cae6d,0x6b09d2bd85728308faa51a74f78b1f27e798a4ff0751...,No,BUY,2026-04-16 00:00:18+00:00,0x540db9a920f5f357a16b5fa60298f56cf43da4830ec2...,0.002,0.002,...,168.380000,0.000000,NaN,None,False,NaN,0.000000,0.000000,0.000000,-0.336760
1,trigger,2,0xad5353afe30c2da57709e2704ef3ccdcf67eef24,0x076ccd097c633ce50d4b9540eee1c006455a4c1d46b7...,Yes,BUY,2026-04-16 00:00:54+00:00,0x2b7bc0e6a5bb576ca9e64ad64b32b65ed1d4481f88cf...,0.390,0.390,...,8.810000,0.000000,NaN,None,False,NaN,0.000000,0.000000,0.000000,-3.435900
3,fill,3,0xad5353afe30c2da57709e2704ef3ccdcf67eef24,0x634ac8f30b4aac7e4be6c968097e4c4708f047e6641d...,No,BUY,2026-04-16 00:03:02+00:00,0x581e713517981708f2c90f8bafd051ec97141d28daf7...,0.570,NaN,...,9.337703,NaN,9.337703,opposite_token,False,-5.608224,9.337703,9.337703,9.337703,NaN
2,trigger,3,0xad5353afe30c2da57709e2704ef3ccdcf67eef24,0x634ac8f30b4aac7e4be6c968097e4c4708f047e6641d...,No,BUY,2026-04-16 00:02:18+00:00,0x804e2e1d62a970d4e07828189248b936aee9797b51e7...,0.600,0.600,...,9.337703,9.337703,NaN,None,False,NaN,0.000000,0.000000,0.000000,-5.602622
4,trigger,4,0xc468b3b8564017fcf2bf9ede2608a0ea24b1009d,0x6f94ad47610b2f7540c94382082fb6f5fc1a5e648d63...,No,BUY,2026-04-16 00:07:16+00:00,0x0ac0827923a47fce87645b4c2b79568dc0de10e54028...,0.953,0.953,...,10.493179,0.000000,NaN,None,True,NaN,0.000000,0.000000,0.000000,2.350000
5,trigger,5,0xad5353afe30c2da57709e2704ef3ccdcf67eef24,0xad01abbcfce6ceb5637db9a3381c243670014d7e2312...,Yes,BUY,2026-04-16 00:09:16+00:00,0x22c8e664d4ef89c687b6bafce2b6129bfc39a07ac0d4...,0.460,0.460,...,7.530000,0.000000,NaN,None,False,NaN,0.000000,0.000000,0.000000,-3.463800


In [21]:
event_ledger[event_ledger['side'] == 'BUY'].groupby('wallet').agg(
    fill_pnl_usdc_sum=('fill_pnl_usdc', 'sum'),
    wallet_trade_pnl_sum=('wallet_trade_pnl', 'sum'),
    buy_count=('side', 'count')
)

,fill_pnl_usdc_sum,wallet_trade_pnl_sum,buy_count
wallet,,,
0x06a0d00aca3a50fc31ff569dad5737347bdb63b9,-60.060000,-12009.335074,18
0x0711e162e05349de3d87626dea4285d08537f03c,95.036910,1864.348005,2853
0x081688cdfa52f2edcc3c2e56c02399314d99377a,-50.473330,-156.805007,148
0x0ad7f3411bc87f0b5362155e638f04ef05700971,-10.933956,100.425775,3267
0x0cb10c40b0776e9ee8cef970af85724654dda76c,-42.046447,1672.790363,1049
0x0dedae6a02ea2ff8018ba5f277632919ed1c9025,125.633199,1754.770566,193
0x160aa7433e68ef858411323b3cfc8248051bb058,0.050362,6.102600,2
0x17c6b26eaeb2a6a2ef768ae0e27c9591e2a08aca,0.422105,217.499990,2
0x1c144e30f405a25f991cbd8baa15d40599090869,-140.330872,-775.111389,648


In [22]:
event_ledger[event_ledger['side'] == 'BUY'].groupby('condition_id').agg(
    fill_pnl_usdc_sum=('fill_pnl_usdc', 'sum'),
    wallet_trade_pnl_sum=('wallet_trade_pnl', 'sum'),
    buy_count=('side', 'count')
).sort_values('fill_pnl_usdc_sum', ascending=False).head(10)

,fill_pnl_usdc_sum,wallet_trade_pnl_sum,buy_count
condition_id,,,
0xaa145cf5714455f546afe53037b8712df749ba96c04820c2d41f37f877f29697,510.207998,38990.418001,1366
0x471e2fa06bd8975b77be49a36781bc41286d57bffa84de55173f4eacdbd52b28,413.232170,4003.731262,450
0x5341e90b72330a8eb96cb685dcffe51ad41783cc12e91b6858b9bd996b3d5b63,344.928149,7737.879181,1008
0x1161c3b9969ad118659e604b2442c858bca0f76d1564e277178e5cf628dac974,166.135136,2232.405222,241
0xc7140ddb5ae5dc94d4553fb05d4600816f33ff024844cebe8326f4c41c4a1a47,161.806832,9126.041966,344
0xd8423c8dc3174c6839b0cf667ce1a1364f85325739409e0f187562227cae44cc,124.087058,21291.782424,35
0xe546672750517f62c45a5a00067481981e62b9c20fa8220203232c9dc8fd2093,123.531655,27557.412509,340
0x2135ffcb43ba9103bb6acf7116d2d5aa98bef6d9eb3dc9c85ea00cb79513f3ec,116.304097,15899.432767,685
0xf9f0965a3bd704ad9c184851242606ad35a4ffe733febf5074d4fbcc59d52eb8,114.042244,-28824.636803,1354


In [23]:
(fills[fills['condition_id'] == '0x3488f31e6449f9803f99a8b5dd232c7ad883637f1c86e6953305a2ef19c77f20'][['dt', 'side', 'outcome',
                                                                                                        'price', 'fill_qty', 'exec_price', 
                                                                                                        # 'filled_wallet_total_position',
                                                                                                          'filled_token_total_position', 
                                                                                                          'filled_token_by_wallet_position', 
                                                                                                          'fill_pnl_usdc',
                                                                                                          'order_id','wallet', 'tx_hash']]
 .sort_values('dt')
 .head(10))

,dt,side,outcome,price,fill_qty,exec_price,filled_token_total_position,filled_token_by_wallet_position,fill_pnl_usdc,order_id,wallet,tx_hash


In [24]:
sample_filled_orders = fills["order_id"].unique()[:50]
event_ledger[
    (event_ledger['side'] == 'SELL') &
    (event_ledger["order_id"].isin(sample_filled_orders))
].sort_values(["order_id", "dt"])

,row_type,order_id,wallet,condition_id,outcome,side,dt,tx_hash,price,order_price,...,fill_qty,fill_source,token_winner,exec_price,fill_pnl_usdc,wallet_trade_pnl,filled_wallet_total_position,filled_token_total_position,filled_token_by_wallet_position,shard
132,trigger,91,0xb40e89677d59665d5188541ad860450a6e2a7cc9,0x724d7eaffc5b26082105d7a9aebd6b729fe0e87368b8...,No,SELL,2026-04-16 02:12:20+00:00,0xba5af2d7d61cd8916283743b8d0e1ab7b441ec9fb969...,0.500000,0.500000,...,NaN,None,True,NaN,NaN,-3.125000,0.380000,0.380000,0.380000,7
137,fill,91,0xb40e89677d59665d5188541ad860450a6e2a7cc9,0x724d7eaffc5b26082105d7a9aebd6b729fe0e87368b8...,No,SELL,2026-04-16 02:12:22+00:00,0x93579f6d9683aea3ba47bf9cb8dbec1c43486356edcf...,0.560000,NaN,...,0.067995,same_token,True,0.500000,-0.034031,NaN,0.312005,0.312005,0.312005,7
138,fill,91,0xb40e89677d59665d5188541ad860450a6e2a7cc9,0x724d7eaffc5b26082105d7a9aebd6b729fe0e87368b8...,No,SELL,2026-04-16 02:12:22+00:00,0x93579f6d9683aea3ba47bf9cb8dbec1c43486356edcf...,0.560000,NaN,...,0.312005,opposite_token,True,0.500000,-0.156159,NaN,0.000000,0.000000,0.000000,7
136,trigger,95,0xb40e89677d59665d5188541ad860450a6e2a7cc9,0x724d7eaffc5b26082105d7a9aebd6b729fe0e87368b8...,No,SELL,2026-04-16 02:12:20+00:00,0x95757ec1bcbc4d2455e112c43f9c35234e3eb86bb9b7...,0.470000,0.470000,...,NaN,None,True,NaN,NaN,-9.937500,0.380000,0.380000,0.380000,7
145,fill,95,0xb40e89677d59665d5188541ad860450a6e2a7cc9,0x724d7eaffc5b26082105d7a9aebd6b729fe0e87368b8...,No,SELL,2026-04-16 02:12:44+00:00,0x497139428b394bc5d8025c5333e555b64e060c6a91b0...,0.560000,NaN,...,0.380000,opposite_token,True,0.470000,-0.201579,NaN,33.050000,0.000000,0.000000,7
152,trigger,99,0xad5353afe30c2da57709e2704ef3ccdcf67eef24,0x724d7eaffc5b26082105d7a9aebd6b729fe0e87368b8...,Yes,SELL,2026-04-16 02:16:26+00:00,0x016b497a7850ceb6b846ed52ce6bdd5fde32e4222306...,0.410000,0.410000,...,NaN,None,False,NaN,NaN,7.802300,34.145814,56.305814,23.255814,7
156,fill,99,0xad5353afe30c2da57709e2704ef3ccdcf67eef24,0x724d7eaffc5b26082105d7a9aebd6b729fe0e87368b8...,Yes,SELL,2026-04-16 02:18:50+00:00,0x976f4aae6053a23cc251a29f32f23d7a402e88a5feab...,0.420000,NaN,...,16.949153,same_token,False,0.410000,6.942203,NaN,17.196661,39.356661,6.306661,7
154,trigger,100,0xad5353afe30c2da57709e2704ef3ccdcf67eef24,0x1b9d9b3651fc12d91ef4eb77d7283b594452081442dd...,Yes,SELL,2026-04-16 02:16:58+00:00,0xc2c854699e18e7db9da45a8083f5f62cc779a22ab9b3...,0.889352,0.889352,...,NaN,None,True,NaN,NaN,-2.674370,10.811915,10.811915,10.811915,1
162,fill,100,0xad5353afe30c2da57709e2704ef3ccdcf67eef24,0x1b9d9b3651fc12d91ef4eb77d7283b594452081442dd...,Yes,SELL,2026-04-16 02:19:18+00:00,0x1e157ab7b19c147261a6b873a115d35cc95209287a30...,0.909000,NaN,...,9.900000,opposite_token,True,0.889352,-1.104223,NaN,0.911915,0.911915,0.911915,1
163,fill,100,0xad5353afe30c2da57709e2704ef3ccdcf67eef24,0x1b9d9b3651fc12d91ef4eb77d7283b594452081442dd...,Yes,SELL,2026-04-16 02:19:18+00:00,0x1e157ab7b19c147261a6b873a115d35cc95209287a30...,0.910000,NaN,...,0.911915,opposite_token,True,0.889352,-0.101713,NaN,0.000000,0.000000,0.000000,1


## Build PnL time series

In [25]:
def resample_hourly_series(df: pd.DataFrame, dt_col: str, pnl_col: str) -> pd.DataFrame:
    """Resample a PnL column to 1-h buckets and compute cumulative sum."""
    if df.empty or pnl_col not in df.columns:
        return pd.DataFrame(columns=["trade_dt", "net_pnl_usdc", "cum_net_pnl_usdc"])
    hourly = (
        df.assign(trade_dt=df[dt_col].dt.floor("1h"))
        .groupby("trade_dt", as_index=False)[pnl_col]
        .sum()
        .rename(columns={pnl_col: "net_pnl_usdc"})
        .sort_values("trade_dt")
        .reset_index(drop=True)
    )
    hourly["cum_net_pnl_usdc"] = hourly["net_pnl_usdc"].cumsum()
    return hourly


def with_zero_anchor(hourly: pd.DataFrame) -> pd.DataFrame:
    """Prepend a zero anchor one hour before the first bucket."""
    if hourly.empty:
        return hourly
    anchor = pd.DataFrame({
        "trade_dt": [hourly["trade_dt"].min() - pd.Timedelta(hours=1)],
        "net_pnl_usdc": [0.0],
        "cum_net_pnl_usdc": [0.0],
    })
    return pd.concat([anchor, hourly], ignore_index=True)


# Wallet cohort PnL: from trigger rows (their actual trade_pnl)
wallet_hourly = resample_hourly_series(triggers, "dt", "wallet_trade_pnl")

# Copy-strategy PnL: from fill rows
copy_hourly = resample_hourly_series(fills, "dt", "fill_pnl_usdc")

print(f"Wallet cohort hourly buckets : {len(wallet_hourly)}")
print(f"Copy strategy hourly buckets : {len(copy_hourly)}")

Wallet cohort hourly buckets : 1217
Copy strategy hourly buckets : 1037


In [26]:
wallet_hourly.head()

,trade_dt,net_pnl_usdc,cum_net_pnl_usdc
0,2026-04-16 00:00:00+00:00,161.817890,161.817890
1,2026-04-16 01:00:00+00:00,172.834692,334.652582
2,2026-04-16 02:00:00+00:00,-1211.127312,-876.474730
3,2026-04-16 03:00:00+00:00,1348.526188,472.051458
4,2026-04-16 04:00:00+00:00,-209.015013,263.036444


## Cumulative PnL chart

In [27]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = go.Figure()

# ── Wallet cohort trace ───────────────────────────────────────────────────────
if not wallet_hourly.empty:
    wh = with_zero_anchor(wallet_hourly)
    fig.add_trace(go.Scatter(
        x=wh["trade_dt"],
        y=wh["cum_net_pnl_usdc"],
        mode="lines",
        line=dict(dash="dot", width=2),
        opacity=0.7,
        name="Watched wallets (raw PnL)",
        hovertemplate=(
            "<b>Watched wallets</b><br>"
            "%{x|%Y-%m-%d %H:%M}<br>"
            "cum PnL: %{y:.2f} USDC<extra></extra>"
        ),
    ))

# ── Copy-strategy trace ───────────────────────────────────────────────────────
if not copy_hourly.empty:
    ch = with_zero_anchor(copy_hourly)
    fig.add_trace(go.Scatter(
        x=ch["trade_dt"],
        y=ch["cum_net_pnl_usdc"],
        mode="lines",
        line=dict(dash="solid", width=2),
        name="Copy strategy (filled)",
        hovertemplate=(
            "<b>Copy strategy</b><br>"
            "%{x|%Y-%m-%d %H:%M}<br>"
            "cum PnL: %{y:.2f} USDC<extra></extra>"
        ),
    ))

fig.add_hline(y=0, line_dash="dash", line_color="grey", line_width=1, opacity=0.5)

fig.update_layout(
    template="plotly_white",
    height=480,
    title=dict(
        text=(
            f"Copy-trade backtest — cumulative PnL (1h) | "
            f"{period_start} → {period_end} | "
            f"{len(WATCHED_WALLETS)} wallets | "
            f"worse_price={WORSE_PRICE_OFFSET:.2f} | "
            f"horizon={int(FILL_HORIZON_SECONDS)}s"
        ),
        font=dict(size=13),
    ),
    xaxis_title="Time (1h buckets)",
    yaxis_title="Cumulative net PnL (USDC)",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
)

fig.show()

## Diagnostics

In [28]:
# Fill rate by side (order-level)
if not triggers.empty:
    trig_by_side = triggers.groupby("side").size().rename("triggers")
    fill_by_side = fills.groupby("side")["order_id"].nunique().rename("orders_filled")
    fill_rows_by_side = fills.groupby("side").size().rename("fill_rows")
    side_summary = pd.concat([trig_by_side, fill_by_side, fill_rows_by_side], axis=1).fillna(0).astype(int)
    side_summary["fill_rate"] = (side_summary["orders_filled"] / side_summary["triggers"] * 100).round(1)
    display(side_summary)


,triggers,orders_filled,fill_rows,fill_rate
side,,,,
BUY,431585,15266,22839,3.5
SELL,16995,10979,15086,64.6


In [29]:
# Fill source breakdown by side
if not fills.empty:
    display(
        fills.groupby(["side", "fill_source"]).size()
        .rename("count")
        .reset_index()
        .sort_values(["side", "fill_source"])
    )

,side,fill_source,count
0,BUY,opposite_token,6197
1,BUY,same_token,16642
2,SELL,opposite_token,6015
3,SELL,same_token,9071


In [30]:
df = event_ledger.assign(
    is_trigger=event_ledger["row_type"].eq("trigger"),
    is_fill=event_ledger["row_type"].eq("fill"),
)

# --- wallet-level stats (row-based) ---
wallet_stats = (
    df.groupby("wallet")
    .agg(
        total_triggers=("is_trigger", "sum"),
        total_fills=("is_fill", "sum"),
        total_fill_pnl=("fill_pnl_usdc", "sum"),
        total_trade_pnl=("wallet_trade_pnl", "sum"),
        total_pnl=("fill_pnl_usdc", "sum"),
    )
)

# --- order-level logic (trigger -> fill) ---
order_stats = (
    df.groupby(["wallet", "order_id"])
    .agg(
        has_trigger=("is_trigger", "any"),
        has_fill=("is_fill", "any"),
    )
    .assign(trigger_with_fill=lambda x: x["has_trigger"] & x["has_fill"])
    .groupby("wallet")
    .agg(triggers_with_fill=("trigger_with_fill", "sum"))
)

# --- combine ---
result = wallet_stats.join(order_stats)

# --- derived metric ---
result["fill_ratio"] = (
    result["triggers_with_fill"] / result["total_triggers"]
).fillna(0)

result.sort_values("total_pnl", ascending=False).head()

,total_triggers,total_fills,total_fill_pnl,total_trade_pnl,total_pnl,triggers_with_fill,fill_ratio
wallet,,,,,,,
0x9238743eeba8bbfe9e85ac7ba2e1e3d77877b73e,807,687,394.217453,1220.324656,394.217453,336,0.416357
0xc34a4af2e3a818fdcdfaa7b5b140eca017ae6007,657,589,256.006091,1978.583036,256.006091,383,0.582953
0xa0bca9bdd8540da95060ed1fafb78aa03835d428,1559,604,235.528006,-5480.265856,235.528006,421,0.270045
0x6df6e2a9ba1e8d7609daada0a83138817f4a8458,125,93,225.159092,11527.738231,225.159092,61,0.488000
0x1c4cd350bce3cb791daf88ce2034de9b03318d85,469,329,190.484839,1687.435989,190.484839,204,0.434968


In [31]:
len(df)

486505

In [32]:
# get diff between trigger and fill timestamp per order_id
# without lambda, using groupby + agg + merge
trigger_df = df[df['row_type'] == 'trigger'].groupby('order_id')['dt'].min().reset_index()
fill_df = df[df['row_type'] == 'fill'].groupby('order_id')['dt'].min().reset_index()

merged_df = pd.merge(trigger_df, fill_df, on='order_id', how='outer', suffixes=('_trigger', '_fill'))
merged_df['diff_dt'] = merged_df['dt_fill'] - merged_df['dt_trigger']

In [33]:
result[result['total_fill_pnl'] < 0].sort_values("total_fill_pnl").index.tolist()

['0x39d0f1dca6fb7e5514858c1a337724a426764fe8',
 '0xfe9339e4eca140c1a16f94a52b5cde4d29768d0e',
 '0xad5353afe30c2da57709e2704ef3ccdcf67eef24',
 '0xcb59ea1babf3b26ea50088057e427ab0ed8f0a22',
 '0x9dfe2f73d3c988a9d69df8fa0beb85651340b3dd',
 '0x945a49252f772a10c6ddd1d1e1e24ee20438a48c',
 '0x969fae0a3a93778adc42178f72c612ed8c4e4d55',
 '0x6640bd87f6e4b6e8d62457448bd1b3a4711a2202',
 '0xfb328b94ed05115259bbc48ba8182df1416edb85',
 '0xe6055a224d277a3eb39df9ee5ea8201c55c4ac5f',
 '0x66c1a6fe836ff555ca32848646acedbbe93bfa3f',
 '0x8f42ae0a01c0383c7ca8bd060b86a645ee74b88f',
 '0xc468b3b8564017fcf2bf9ede2608a0ea24b1009d',
 '0x41583f2efc720b8e2682750fffb67f2806fece9f',
 '0x06a0d00aca3a50fc31ff569dad5737347bdb63b9',
 '0xb40e89677d59665d5188541ad860450a6e2a7cc9',
 '0x081688cdfa52f2edcc3c2e56c02399314d99377a',
 '0x1c144e30f405a25f991cbd8baa15d40599090869',
 '0x9783178832982d420baa733d2ec29b020eb9264f',
 '0x596a78ff6de24b12fd39f827aecf93975e2f7a0f',
 '0xc412fdbe9956c2e242e8b45fae131c3e8598592d',
 '0x7656ed7f5

In [34]:
# Per-wallet PnL breakdown
if not fills.empty:
    wallet_pnl_df = (
        fills.groupby("wallet")
        .agg(
            filled_orders=("order_id", "nunique"),
            fill_rows=("order_id", "count"),
            net_pnl_usdc=("fill_pnl_usdc", "sum"),
            total_qty=("fill_qty", "sum"),
            win_rate=("token_winner", lambda s: fills.loc[s.index].groupby("order_id")["token_winner"].first().mean()),
        )
        .assign(win_rate=lambda d: (d["win_rate"] * 100).round(1))
        .sort_values("net_pnl_usdc", ascending=False)
        .reset_index()
    )
    display(wallet_pnl_df)


,wallet,filled_orders,fill_rows,net_pnl_usdc,total_qty,win_rate
0,0x9238743eeba8bbfe9e85ac7ba2e1e3d77877b73e,336,687,394.217453,4929.891613,70.5
1,0xc34a4af2e3a818fdcdfaa7b5b140eca017ae6007,383,589,256.006091,5569.430596,67.1
2,0xa0bca9bdd8540da95060ed1fafb78aa03835d428,421,604,235.528006,5019.686895,50.4
3,0x6df6e2a9ba1e8d7609daada0a83138817f4a8458,61,93,225.159092,1026.283362,75.4
4,0x1c4cd350bce3cb791daf88ce2034de9b03318d85,204,329,190.484839,3208.026918,24.5
5,0x74cbe13dba27a6a16805e9e7142ee68aa09cae6d,115,164,184.666500,1923.802030,28.7
6,0xca12d28728c46d3484395243f5f842f2c321ea03,108,143,176.681042,1717.250731,51.9
7,0xe732156a2d84cdfb4de831d3f11a22899e49898f,259,426,139.163083,4016.827161,43.2
8,0x3b4484b6c8cbfdaa383ba337ab3f0d71055e264e,33,83,134.563297,964.033451,39.4
9,0x9b2fcb79bdb8a6a6174da327e9335f7889c6d8f8,170,284,120.198825,2846.482634,62.9


In [35]:
# Orders that were NOT filled at all
filled_order_ids = set(fills["order_id"].unique()) if not fills.empty else set()
unfilled_triggers = triggers[~triggers["order_id"].isin(filled_order_ids)]
print(f"Unfilled orders    : {len(unfilled_triggers):,} ({len(unfilled_triggers)/max(len(triggers),1)*100:.1f}% of all triggers)")

# Partially filled orders (filled but qty_filled < qty_ordered)
if not fills.empty:
    filled_trig = triggers[triggers["order_id"].isin(filled_order_ids)].copy()
    partial_mask = filled_trig["qty_filled"] < filled_trig["qty_ordered"] - 1e-9
    partial_fills = filled_trig[partial_mask]
    print(f"Partially filled   : {len(partial_fills):,} ({len(partial_fills)/max(len(filled_trig),1)*100:.1f}% of filled orders)")
    print(f"Fully filled       : {len(filled_trig)-len(partial_fills):,}")

if not unfilled_triggers.empty:
    print("\nUnfilled breakdown by side:")
    display(unfilled_triggers.groupby("side").size().rename("count").reset_index())


Unfilled orders    : 422,335 (94.1% of all triggers)
Partially filled   : 6,665 (25.4% of filled orders)
Fully filled       : 19,580

Unfilled breakdown by side:


,side,count
0,BUY,416319
1,SELL,6016


In [36]:
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 100)
fills[fills['order_id'] == 34]

,row_type,order_id,wallet,condition_id,outcome,side,dt,tx_hash,price,order_price,...,fill_qty,fill_source,token_winner,exec_price,fill_pnl_usdc,wallet_trade_pnl,filled_wallet_total_position,filled_token_total_position,filled_token_by_wallet_position,shard
52,fill,34,0xc34a4af2e3a818fdcdfaa7b5b140eca017ae6007,0xc7140ddb5ae5dc94d4553fb05d4600816f33ff024844cebe8326f4c41c4a1a47,Yes,BUY,2026-04-16 00:51:38+00:00,0x415d56c37624ae4cf3cada43b7e9f59a59e79d5a60138f9cac7a992f544fb280,0.689,NaN,...,3.215434,same_token,True,0.689,0.997784,NaN,16.077169,30.718457,16.077169,c


In [37]:
fills.groupby("wallet")["fill_pnl_usdc"].sum().sort_values(ascending=False, key=abs).head(100)

wallet
0x9238743eeba8bbfe9e85ac7ba2e1e3d77877b73e    394.217453
0x39d0f1dca6fb7e5514858c1a337724a426764fe8   -374.491532
0xfe9339e4eca140c1a16f94a52b5cde4d29768d0e   -264.914427
0xc34a4af2e3a818fdcdfaa7b5b140eca017ae6007    256.006091
0xad5353afe30c2da57709e2704ef3ccdcf67eef24   -246.520833
0xa0bca9bdd8540da95060ed1fafb78aa03835d428    235.528006
0xcb59ea1babf3b26ea50088057e427ab0ed8f0a22   -233.638273
0x9dfe2f73d3c988a9d69df8fa0beb85651340b3dd   -233.295774
0x6df6e2a9ba1e8d7609daada0a83138817f4a8458    225.159092
0x945a49252f772a10c6ddd1d1e1e24ee20438a48c   -205.153563
0x1c4cd350bce3cb791daf88ce2034de9b03318d85    190.484839
0x74cbe13dba27a6a16805e9e7142ee68aa09cae6d    184.666500
0x969fae0a3a93778adc42178f72c612ed8c4e4d55   -180.603185
0xca12d28728c46d3484395243f5f842f2c321ea03    176.681042
0x6640bd87f6e4b6e8d62457448bd1b3a4711a2202   -144.063643
0xe732156a2d84cdfb4de831d3f11a22899e49898f    139.163083
0x3b4484b6c8cbfdaa383ba337ab3f0d71055e264e    134.563297
0xfb328b94ed05115259bbc4

In [38]:
trigger_wallets = triggers[["order_id", "wallet"]].set_index("order_id")["wallet"]

fills_with_wallet = fills.merge(trigger_wallets, on="order_id", how="left", suffixes=("", "_trigger"))

fills_with_wallet['notional'] = fills_with_wallet['fill_qty'] * fills_with_wallet['exec_price']

fills_with_wallet.groupby(["wallet", "condition_id"]).agg(
    pnl=("fill_pnl_usdc", "sum"),
    orders_filled=("order_id", "nunique")
    ).sort_values(['wallet', "pnl"], ascending=[True, False]).head(1)

,,pnl,orders_filled
wallet,condition_id,,
0x06a0d00aca3a50fc31ff569dad5737347bdb63b9,0xa93b28a6384aefff4e7d5adb2061c444e4a0e95b8ad17755f9cee123c7099b35,-60.06,6


In [39]:
fills_with_wallet.head()

,row_type,order_id,wallet,condition_id,outcome,side,dt,tx_hash,price,order_price,...,token_winner,exec_price,fill_pnl_usdc,wallet_trade_pnl,filled_wallet_total_position,filled_token_total_position,filled_token_by_wallet_position,shard,wallet_trigger,notional
0,fill,3,0xad5353afe30c2da57709e2704ef3ccdcf67eef24,0x634ac8f30b4aac7e4be6c968097e4c4708f047e6641d005d3a0948c45f0912d3,No,BUY,2026-04-16 00:03:02+00:00,0x581e713517981708f2c90f8bafd051ec97141d28daf795a338e4f16dde2d6654,0.57,NaN,...,False,0.60,-5.608224,NaN,9.337703,9.337703,9.337703,6,0xad5353afe30c2da57709e2704ef3ccdcf67eef24,5.602622
1,fill,6,0xad5353afe30c2da57709e2704ef3ccdcf67eef24,0x01aaece056f8163129041a6a8e0fcfca7ca331df1ca652f8084ce60b46e6d870,No,BUY,2026-04-16 00:16:08+00:00,0x044f6bfd34f6bbf911854b415cb2f2431972f055c4235fbaa55ef4a138cc423c,0.67,NaN,...,False,0.71,-2.153666,NaN,3.030302,3.030302,3.030302,0,0xad5353afe30c2da57709e2704ef3ccdcf67eef24,2.151515
2,fill,9,0xad5353afe30c2da57709e2704ef3ccdcf67eef24,0x2c6ac9c7c96fd79079f71d5c1c1a77e9ec2265ff4355dbdaa07e3adeeae6524b,No,BUY,2026-04-16 00:19:26+00:00,0xeacb74d05e1a588d36f0441ff6026fd092178176b88e9669909387f41f53eb3f,0.80,NaN,...,True,0.84,0.867422,NaN,5.450000,5.450000,5.450000,2,0xad5353afe30c2da57709e2704ef3ccdcf67eef24,4.578000
3,fill,9,0xad5353afe30c2da57709e2704ef3ccdcf67eef24,0x2c6ac9c7c96fd79079f71d5c1c1a77e9ec2265ff4355dbdaa07e3adeeae6524b,No,BUY,2026-04-16 00:19:26+00:00,0xeacb74d05e1a588d36f0441ff6026fd092178176b88e9669909387f41f53eb3f,0.79,NaN,...,True,0.84,0.730843,NaN,10.041875,10.041875,10.041875,2,0xad5353afe30c2da57709e2704ef3ccdcf67eef24,3.857175
4,fill,10,0xad5353afe30c2da57709e2704ef3ccdcf67eef24,0x7aa6ecf3de5537b7822a0d6041b713ecc4d8b5c9f58449e16509c8af43db86ef,Yes,BUY,2026-04-16 00:20:54+00:00,0xf7fe728c8cd56d6aeb42337e0a755f76b8915ee986a4ff016ab5817cfb1e06ae,0.82,NaN,...,False,0.82,-4.169766,NaN,5.080000,5.080000,5.080000,7,0xad5353afe30c2da57709e2704ef3ccdcf67eef24,4.165600


In [40]:
wallet_pnls = (
    fills_with_wallet
    .groupby(["wallet", "condition_id"])
    .agg(
        pnl=("fill_pnl_usdc", "sum"),
        orders_filled=("order_id", "nunique"),
        notional=("notional", "sum"),
    )
    .groupby("wallet")
    .agg(
        pnl=("pnl", "sum"),
        notional=("notional", "sum"),
        unique_conditions=("pnl", "size"),  # ✅ count rows
        total_orders_filled=("orders_filled", "sum"),
    )
    .sort_values("pnl", ascending=False)
)

In [41]:
wallet_pnls.head()

,pnl,notional,unique_conditions,total_orders_filled
wallet,,,,
0x9238743eeba8bbfe9e85ac7ba2e1e3d77877b73e,394.217453,2565.640265,45,336
0xc34a4af2e3a818fdcdfaa7b5b140eca017ae6007,256.006091,3467.037537,43,383
0xa0bca9bdd8540da95060ed1fafb78aa03835d428,235.528006,2075.210950,86,421
0x6df6e2a9ba1e8d7609daada0a83138817f4a8458,225.159092,431.996729,12,61
0x1c4cd350bce3cb791daf88ce2034de9b03318d85,190.484839,936.751240,33,204


In [42]:
MAX_WALLET_EXPOSURE = 1000.0  # USDC
wallet_pnls['pnl_limited'] = np.where(
    wallet_pnls['notional'] <= MAX_WALLET_EXPOSURE,
    wallet_pnls['pnl'],
    wallet_pnls['pnl'] * MAX_WALLET_EXPOSURE / wallet_pnls['notional']
)

In [43]:
wallet_pnls.head(10)

,pnl,notional,unique_conditions,total_orders_filled,pnl_limited
wallet,,,,,
0x9238743eeba8bbfe9e85ac7ba2e1e3d77877b73e,394.217453,2565.640265,45,336,153.652660
0xc34a4af2e3a818fdcdfaa7b5b140eca017ae6007,256.006091,3467.037537,43,383,73.840011
0xa0bca9bdd8540da95060ed1fafb78aa03835d428,235.528006,2075.210950,86,421,113.495935
0x6df6e2a9ba1e8d7609daada0a83138817f4a8458,225.159092,431.996729,12,61,225.159092
0x1c4cd350bce3cb791daf88ce2034de9b03318d85,190.484839,936.751240,33,204,190.484839
0x74cbe13dba27a6a16805e9e7142ee68aa09cae6d,184.666500,348.991741,21,115,184.666500
0xca12d28728c46d3484395243f5f842f2c321ea03,176.681042,610.328601,51,108,176.681042
0xe732156a2d84cdfb4de831d3f11a22899e49898f,139.163083,1800.823231,61,259,77.277481
0x3b4484b6c8cbfdaa383ba337ab3f0d71055e264e,134.563297,184.072921,18,33,134.563297


In [44]:
result = (
    fills_with_wallet
    # 1️⃣ PnL per market (condition) per wallet
    .groupby(["wallet", "condition_id"])["fill_pnl_usdc"]
    .sum()
    
    # 2️⃣ Then per wallet
    .groupby("wallet")
    .agg(
        unique_conditions="count",   # number of markets traded
        median_market_pnl="median",  # median PnL across markets
    )
    .sort_values("unique_conditions", ascending=False)
    .head(20)
)

In [45]:
result.sort_values("median_market_pnl", ascending=False).head(100)

,unique_conditions,median_market_pnl
wallet,,
0x9cdda449d7cedf1072d74982a5dc2349df3d3e97,80,0.381416
0x7c3db723f1d4d8cb9c550095203b686cb11e5c6b,80,0.341391
0x22dbd51e1a4e10fff072fa24300238c64033190f,79,0.317296
0xc468b3b8564017fcf2bf9ede2608a0ea24b1009d,66,0.315831
0x9783178832982d420baa733d2ec29b020eb9264f,97,0.194082
0xa0bca9bdd8540da95060ed1fafb78aa03835d428,86,0.185814
0x3c760e45728e0cb4b68c52980a070314b95fd4ee,78,0.143506
0x66c1a6fe836ff555ca32848646acedbbe93bfa3f,63,0.091010
0x49161f6833d0be67e00b0ed06477536134f4f56f,134,-0.025986


In [46]:
fills_with_wallet.head()

,row_type,order_id,wallet,condition_id,outcome,side,dt,tx_hash,price,order_price,...,token_winner,exec_price,fill_pnl_usdc,wallet_trade_pnl,filled_wallet_total_position,filled_token_total_position,filled_token_by_wallet_position,shard,wallet_trigger,notional
0,fill,3,0xad5353afe30c2da57709e2704ef3ccdcf67eef24,0x634ac8f30b4aac7e4be6c968097e4c4708f047e6641d005d3a0948c45f0912d3,No,BUY,2026-04-16 00:03:02+00:00,0x581e713517981708f2c90f8bafd051ec97141d28daf795a338e4f16dde2d6654,0.57,NaN,...,False,0.60,-5.608224,NaN,9.337703,9.337703,9.337703,6,0xad5353afe30c2da57709e2704ef3ccdcf67eef24,5.602622
1,fill,6,0xad5353afe30c2da57709e2704ef3ccdcf67eef24,0x01aaece056f8163129041a6a8e0fcfca7ca331df1ca652f8084ce60b46e6d870,No,BUY,2026-04-16 00:16:08+00:00,0x044f6bfd34f6bbf911854b415cb2f2431972f055c4235fbaa55ef4a138cc423c,0.67,NaN,...,False,0.71,-2.153666,NaN,3.030302,3.030302,3.030302,0,0xad5353afe30c2da57709e2704ef3ccdcf67eef24,2.151515
2,fill,9,0xad5353afe30c2da57709e2704ef3ccdcf67eef24,0x2c6ac9c7c96fd79079f71d5c1c1a77e9ec2265ff4355dbdaa07e3adeeae6524b,No,BUY,2026-04-16 00:19:26+00:00,0xeacb74d05e1a588d36f0441ff6026fd092178176b88e9669909387f41f53eb3f,0.80,NaN,...,True,0.84,0.867422,NaN,5.450000,5.450000,5.450000,2,0xad5353afe30c2da57709e2704ef3ccdcf67eef24,4.578000
3,fill,9,0xad5353afe30c2da57709e2704ef3ccdcf67eef24,0x2c6ac9c7c96fd79079f71d5c1c1a77e9ec2265ff4355dbdaa07e3adeeae6524b,No,BUY,2026-04-16 00:19:26+00:00,0xeacb74d05e1a588d36f0441ff6026fd092178176b88e9669909387f41f53eb3f,0.79,NaN,...,True,0.84,0.730843,NaN,10.041875,10.041875,10.041875,2,0xad5353afe30c2da57709e2704ef3ccdcf67eef24,3.857175
4,fill,10,0xad5353afe30c2da57709e2704ef3ccdcf67eef24,0x7aa6ecf3de5537b7822a0d6041b713ecc4d8b5c9f58449e16509c8af43db86ef,Yes,BUY,2026-04-16 00:20:54+00:00,0xf7fe728c8cd56d6aeb42337e0a755f76b8915ee986a4ff016ab5817cfb1e06ae,0.82,NaN,...,False,0.82,-4.169766,NaN,5.080000,5.080000,5.080000,7,0xad5353afe30c2da57709e2704ef3ccdcf67eef24,4.165600


In [47]:
fills_with_wallet[fills_with_wallet['condition_id'] == '0xfdd729d85e44a153ae8d53e709ced74f249815e93570b98c6037ac479ec257b8'].groupby('wallet')['wallet'].count()

Series([], Name: wallet, dtype: int64)

In [48]:
fills_with_wallet[fills_with_wallet['dt'] >= '2026-05-20'].groupby("condition_id")['condition_id'].count().sort_values(ascending=False).head(20)

condition_id
0x6a8cfe84d17693425f27831db5949d7511f3393d4624b182ac6956164cd32b10    322
0x421bc1929df1429cf2cb94f80c1ce6a3ed0d1f0b7a2749b9890075f94eb549e9    272
0x4ba348328e4d4ddee9e6734c9a369b2e8138611651f9f6dc8f59dea51df6c734    101
0xf526b2fbff94ae996818e76c8b54cc99ffbfa0a1a9972a48253ada65e32786f7     94
0x9a87a0011494682e4d17a9073059cbbd32ebea4bbc7284219afb5e9a6b4ca8f6     76
0x71c49500d051a9b32b9aa5bc22e0a6925a8b2282840c3e62e0a3afcd0403ba03     72
0x83831604f88a889311dc484abc0d327d96f494bba9d41084bf1b84d8a991e97b     64
0x53098776eb95dbbd2e899c4fcd787a043ba725f55433948da2fbd00abdfcd3ae     60
0xffb8a1a5c71eacf9b8dbe4f10b846c239c74680756cb744fcfc760db45eb256e     56
0xcef981d46a1039b6ae02f578d2208302a8a3d63465d24363a1d65a86835a1ae8     56
0xdb22a7749b831aa07a52cbc83213e6c8ceb88226b224a831512f4460011bb0a1     49
0x59a37ea3830d532957b04d3c437a329e14a5dc840096d48c7ee4b55ba3d9cca8     47
0x8374c773aa911e00a8d5c02e0130d57d18a9b5035b26e50f0e48ef1b74dd5c75     44
0xb2e7dd21dd889b9b5644785

In [49]:
from polymarket_analysis.data.data_catalogue import load_markets_processed
mdf = load_markets_processed()
mdf = mdf[
    ~(mdf['primary_tag'].isin(['Sports', 'Crypto']))
    & (mdf['winner_token_id'].notna())
    ].copy().reset_index(drop=True)


In [50]:
mdf.head()

,condition_id,end_date_iso,question,tags,primary_tag,winner_token_id,outcome
0,0x055176c9ebd8775c281ad540d5c16b3323e316507aef2d45c84f061cbc6bbcdc,2022-08-21T00:00:00Z,"MLB: Who will win Boston Red Sox v. Baltimore Orioles, scheduled for August 21, 7:10 PM ET?",[All],All,9612890763764062692282935414227141810568206972440321500296202304471805951204,Orioles
1,0x1409177eae7f1b0406bc0eea9d2715c9f76d88ec7765aed03cf39d59f36008f7,2023-01-06T00:00:00Z,Will a House Speaker be elected by end of Thursday?,[All],All,56419231299380534710656783098291368308638365867243518354348835265264811381897,No
2,0xd6165c1a30269d0799b27ee6ea90c048dfd6b27276cbd2aa7e18fe1c4562faa8,2023-01-14T00:00:00Z,Will Benjamin Netanyahu be the 2023 TIME Person of the Year?,"[Politics, Joe Biden, time magazine, person of the year, Awards, technology, news, future predic...",Politics,115040502429502340199300991115762950152511906206476760034830683863926486402417,No
3,0x9da76626846b7a0f7c5388feb280e42a67564ea104123020fbf59a641b179623,2023-01-14T00:00:00Z,Will Xi Jinping be the 2023 TIME Person of the Year?,"[Politics, Joe Biden, time magazine, person of the year, Awards, technology, news, future predic...",Politics,55070293174349524004776076543655661478039361630374630352242580072992541740735,No
4,0xe7d21325e5cfccbdaddf6ab6ef9bf477cf3aa6615dd36c8c408da99a1dd83237,2023-01-14T00:00:00Z,Will Volodymyr Zelenskyy be the 2023 TIME Person of the Year?,"[Politics, Joe Biden, time magazine, person of the year, Awards, technology, news, future predic...",Politics,5775128592397269721112080310923554806977502333079078864830955469822139503940,No


## Browser plots: PnL and positions

Interactive Plotly charts for cumulative PnL and signed positions. Market hovers include `end_date_iso` from `mdf`.


In [51]:
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

pio.renderers.default = "browser"

TOP_WALLETS_PNL = 10
TOP_MARKETS_PNL = 10
TOP_WALLETS_POSITION = 10
TOP_MARKETS_POSITION = 10

def _short_wallet(wallet: str) -> str:
    wallet = str(wallet)
    return f"{wallet[:6]}...{wallet[-4:]}" if len(wallet) > 14 else wallet

def _short_text(text: str, width: int = 72) -> str:
    if pd.isna(text):
        return "Unknown market"
    text = " ".join(str(text).split())
    return text if len(text) <= width else text[: width - 3] + "..."

def _build_total_ts(df: pd.DataFrame, value_col: str, net_col: str, cum_col: str) -> pd.DataFrame:
    total = (
        df.sort_values("dt")
        .groupby("dt", as_index=False)[value_col]
        .sum()
        .rename(columns={value_col: net_col})
    )
    total[cum_col] = total[net_col].cumsum()
    return total

def _build_position_snapshots(df: pd.DataFrame) -> pd.DataFrame:
    series_cols = [
        "wallet",
        "wallet_label",
        "condition_id",
        "market_label",
        "end_date_iso",
        "market_close_dt",
        "outcome",
    ]

    active = df[
        df["market_close_dt"].isna() | (df["dt"] < df["market_close_dt"])
    ].copy()

    snapshots = (
        active.sort_values(series_cols + ["dt", "order_id"])
        .groupby(series_cols + ["dt"], as_index=False, dropna=False)
        .agg(
            token_position=("filled_token_by_wallet_position", "last"),
            exec_price=("exec_price", "last"),
        )
    )
    snapshots["usdc_position"] = snapshots["token_position"] * snapshots["exec_price"]

    if snapshots.empty:
        snapshots["token_delta"] = pd.Series(dtype=float)
        snapshots["usdc_delta"] = pd.Series(dtype=float)
        return snapshots

    reset = (
        snapshots.groupby(series_cols, as_index=False, dropna=False)
        .agg(
            token_position=("token_position", "last"),
            usdc_position=("usdc_position", "last"),
        )
    )
    reset = reset[
        reset["market_close_dt"].notna()
        & ((reset["token_position"].abs() > 1e-12) | (reset["usdc_position"].abs() > 1e-12))
    ].copy()
    reset["dt"] = reset["market_close_dt"]
    reset["token_position"] = 0.0
    reset["usdc_position"] = 0.0

    combined = pd.concat(
        [
            snapshots[series_cols + ["dt", "token_position", "usdc_position"]],
            reset[series_cols + ["dt", "token_position", "usdc_position"]],
        ],
        ignore_index=True,
    )
    combined = (
        combined.sort_values(series_cols + ["dt"])
        .drop_duplicates(subset=series_cols + ["dt"], keep="last")
        .reset_index(drop=True)
    )
    combined["token_delta"] = combined.groupby(series_cols, dropna=False)["token_position"].diff()
    combined["usdc_delta"] = combined.groupby(series_cols, dropna=False)["usdc_position"].diff()
    combined["token_delta"] = combined["token_delta"].fillna(combined["token_position"])
    combined["usdc_delta"] = combined["usdc_delta"].fillna(combined["usdc_position"])
    return combined

market_meta = (
    mdf[["condition_id", "question", "end_date_iso", "primary_tag"]]
    .drop_duplicates(subset=["condition_id"])
    .copy()
)
market_meta["end_date"] = pd.to_datetime(market_meta["end_date_iso"], utc=True, errors="coerce")
market_meta["market_close_dt"] = market_meta["end_date"].dt.floor("D") + pd.Timedelta(days=1)

plot_fills = fills.copy()
plot_fills["dt"] = pd.to_datetime(plot_fills["dt"], utc=True)
plot_fills = plot_fills.merge(market_meta, on="condition_id", how="left")
plot_fills["wallet_label"] = plot_fills["wallet"].map(_short_wallet)
plot_fills["market_label"] = plot_fills["question"].map(_short_text)
plot_fills["market_label"] = np.where(
    plot_fills["market_label"].eq("Unknown market"),
    plot_fills["condition_id"].str[:12],
    plot_fills["market_label"],
)

pnl_total_ts = _build_total_ts(plot_fills, "fill_pnl_usdc", "net_pnl_usdc", "cum_pnl_usdc")

wallet_pnl_totals = (
    plot_fills.groupby(["wallet", "wallet_label"], as_index=False, dropna=False)
    .agg(total_pnl_usdc=("fill_pnl_usdc", "sum"))
)
top_wallets_pnl = wallet_pnl_totals.reindex(
    wallet_pnl_totals["total_pnl_usdc"].abs().sort_values(ascending=False).index
).head(TOP_WALLETS_PNL).copy()
wallet_pnl_ts = (
    plot_fills[plot_fills["wallet"].isin(top_wallets_pnl["wallet"])]
    .groupby(["wallet", "wallet_label", "dt"], as_index=False, dropna=False)
    .agg(net_pnl_usdc=("fill_pnl_usdc", "sum"))
    .sort_values(["wallet", "dt"])
)

plot_fills["fill_pnl_usdc"] = pd.to_numeric(plot_fills["fill_pnl_usdc"], errors="coerce").fillna(0.0)
wallet_pnl_ts["cum_pnl_usdc"] = plot_fills["fill_pnl_usdc"]

market_pnl_totals = (
    plot_fills.groupby(["condition_id", "market_label", "end_date_iso"], as_index=False, dropna=False)
    .agg(total_pnl_usdc=("fill_pnl_usdc", "sum"))
)
top_markets_pnl = market_pnl_totals.reindex(
    market_pnl_totals["total_pnl_usdc"].abs().sort_values(ascending=False).index
).head(TOP_MARKETS_PNL).copy()
market_pnl_ts = (
    plot_fills[plot_fills["condition_id"].isin(top_markets_pnl["condition_id"])]
    .groupby(["condition_id", "market_label", "end_date_iso", "dt"], as_index=False, dropna=False)
    .agg(net_pnl_usdc=("fill_pnl_usdc", "sum"))
    .sort_values(["condition_id", "dt"])
)
market_pnl_ts["cum_pnl_usdc"] = market_pnl_ts.groupby("condition_id", dropna=False)["net_pnl_usdc"].cumsum()

granular_position_ts = _build_position_snapshots(plot_fills)

wallet_position_all = (
    granular_position_ts.groupby(["wallet", "wallet_label", "dt"], as_index=False, dropna=False)
    .agg(
        net_token_position=("token_delta", "sum"),
        net_usdc_position=("usdc_delta", "sum"),
    )
    .sort_values(["wallet", "dt"])
)
wallet_position_all["cum_token_position"] = wallet_position_all.groupby("wallet", dropna=False)["net_token_position"].cumsum()
wallet_position_all["cum_usdc_position"] = wallet_position_all.groupby("wallet", dropna=False)["net_usdc_position"].cumsum()
wallet_position_rank = (
    wallet_position_all.groupby(["wallet", "wallet_label"], as_index=False, dropna=False)
    .agg(max_abs_usdc_position=("cum_usdc_position", lambda s: s.abs().max()))
)
top_wallets_position = wallet_position_rank.sort_values("max_abs_usdc_position", ascending=False).head(TOP_WALLETS_POSITION).copy()
wallet_position_ts = wallet_position_all[wallet_position_all["wallet"].isin(top_wallets_position["wallet"])].copy()

market_position_all = (
    granular_position_ts.groupby(["condition_id", "market_label", "end_date_iso", "dt"], as_index=False, dropna=False)
    .agg(
        net_token_position=("token_delta", "sum"),
        net_usdc_position=("usdc_delta", "sum"),
    )
    .sort_values(["condition_id", "dt"])
)
market_position_all["cum_token_position"] = market_position_all.groupby("condition_id", dropna=False)["net_token_position"].cumsum()
market_position_all["cum_usdc_position"] = market_position_all.groupby("condition_id", dropna=False)["net_usdc_position"].cumsum()
market_position_rank = (
    market_position_all.groupby(["condition_id", "market_label", "end_date_iso"], as_index=False, dropna=False)
    .agg(max_abs_usdc_position=("cum_usdc_position", lambda s: s.abs().max()))
)
top_markets_position = market_position_rank.sort_values("max_abs_usdc_position", ascending=False).head(TOP_MARKETS_POSITION).copy()
market_position_ts = market_position_all[market_position_all["condition_id"].isin(top_markets_position["condition_id"])].copy()

token_total_ts = _build_total_ts(granular_position_ts, "token_delta", "net_token_position", "cum_token_position")
usdc_total_ts = _build_total_ts(granular_position_ts, "usdc_delta", "net_usdc_position", "cum_usdc_position")

print(
    f"Prepared plot datasets: total pnl={len(pnl_total_ts)}, "
    f"wallet pnl series={wallet_pnl_ts['wallet'].nunique()}, "
    f"market pnl series={market_pnl_ts['condition_id'].nunique()}, "
    f"wallet position series={wallet_position_ts['wallet'].nunique()}, "
    f"market position series={market_position_ts['condition_id'].nunique()}"
)


Prepared plot datasets: total pnl=23955, wallet pnl series=10, market pnl series=10, wallet position series=10, market position series=10


In [52]:
fig_pnl = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.05,
    subplot_titles=(
        "Total cumulative PnL",
        f"Per-wallet cumulative PnL (top {TOP_WALLETS_PNL} by abs total PnL)",
        f"Per-market cumulative PnL (top {TOP_MARKETS_PNL} by abs total PnL)",
    ),
)

fig_pnl.add_trace(
    go.Scatter(
        x=pnl_total_ts["dt"],
        y=pnl_total_ts["cum_pnl_usdc"],
        mode="lines",
        name="Total",
        line=dict(width=3, color="#1f77b4"),
        hovertemplate="%{x|%Y-%m-%d %H:%M}<br>Total cum PnL: %{y:,.2f} USDC<extra></extra>",
    ),
    row=1,
    col=1,
)

wallet_totals_map = top_wallets_pnl.set_index("wallet")["total_pnl_usdc"].to_dict()
for wallet in top_wallets_pnl["wallet"]:
    sub = wallet_pnl_ts[wallet_pnl_ts["wallet"] == wallet]
    label = sub["wallet_label"].iloc[0]
    total = wallet_totals_map[wallet]
    fig_pnl.add_trace(
        go.Scatter(
            x=sub["dt"],
            y=sub["cum_pnl_usdc"],
            mode="lines",
            name=f"{label} ({total:,.1f})",
            hovertemplate=(
                f"Wallet: {wallet}<br>"
                + "%{x|%Y-%m-%d %H:%M}<br>"
                + "Cum PnL: %{y:,.2f} USDC<extra></extra>"
            ),
        ),
        row=2,
        col=1,
    )

market_totals_map = top_markets_pnl.set_index("condition_id")["total_pnl_usdc"].to_dict()
for condition_id in top_markets_pnl["condition_id"]:
    sub = market_pnl_ts[market_pnl_ts["condition_id"] == condition_id]
    label = sub["market_label"].iloc[0]
    end_date_iso = sub["end_date_iso"].iloc[0]
    total = market_totals_map[condition_id]
    fig_pnl.add_trace(
        go.Scatter(
            x=sub["dt"],
            y=sub["cum_pnl_usdc"],
            mode="lines",
            name=f"{label} ({total:,.1f})",
            hovertemplate=(
                f"Market: {label}<br>"
                + f"end_date_iso: {end_date_iso}<br>"
                + f"condition_id: {condition_id}<br>"
                + "%{x|%Y-%m-%d %H:%M}<br>"
                + "Cum PnL: %{y:,.2f} USDC<extra></extra>"
            ),
        ),
        row=3,
        col=1,
    )

for row in (1, 2, 3):
    fig_pnl.add_hline(y=0, line_dash="dash", line_color="grey", line_width=1, opacity=0.5, row=row, col=1)

fig_pnl.update_yaxes(title_text="USDC", row=1, col=1)
fig_pnl.update_yaxes(title_text="USDC", row=2, col=1)
fig_pnl.update_yaxes(title_text="USDC", row=3, col=1)
fig_pnl.update_xaxes(title_text="Time", row=3, col=1)
fig_pnl.update_layout(
    template="plotly_white",
    height=1200,
    title="Stage 2 copy-trade backtest - cumulative PnL",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
)
fig_pnl.show(renderer="browser")


In [53]:
fig_token_pos = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.05,
    subplot_titles=(
        "Total signed token position",
        f"Per-wallet signed token position (top {TOP_WALLETS_POSITION} by max abs USDC position)",
        f"Per-market signed token position (top {TOP_MARKETS_POSITION} by max abs USDC position)",
    ),
)

fig_token_pos.add_trace(
    go.Scatter(
        x=token_total_ts["dt"],
        y=token_total_ts["cum_token_position"],
        mode="lines",
        line_shape="hv",
        name="Total token position",
        line=dict(width=3, color="#2ca02c"),
        hovertemplate="%{x|%Y-%m-%d %H:%M}<br>Signed token position: %{y:,.2f}<extra></extra>",
    ),
    row=1,
    col=1,
)

for wallet in top_wallets_position["wallet"]:
    sub = wallet_position_ts[wallet_position_ts["wallet"] == wallet]
    label = sub["wallet_label"].iloc[0]
    max_abs = sub["cum_usdc_position"].abs().max()
    fig_token_pos.add_trace(
        go.Scatter(
            x=sub["dt"],
            y=sub["cum_token_position"],
            mode="lines",
            line_shape="hv",
            name=f"{label} ({max_abs:,.1f} USDC)",
            hovertemplate=(
                f"Wallet: {wallet}<br>"
                + "%{x|%Y-%m-%d %H:%M}<br>"
                + "Signed token position: %{y:,.2f}<extra></extra>"
            ),
        ),
        row=2,
        col=1,
    )

for condition_id in top_markets_position["condition_id"]:
    sub = market_position_ts[market_position_ts["condition_id"] == condition_id]
    label = sub["market_label"].iloc[0]
    end_date_iso = sub["end_date_iso"].iloc[0]
    max_abs = sub["cum_usdc_position"].abs().max()
    fig_token_pos.add_trace(
        go.Scatter(
            x=sub["dt"],
            y=sub["cum_token_position"],
            mode="lines",
            line_shape="hv",
            name=f"{label} ({max_abs:,.1f} USDC)",
            hovertemplate=(
                f"Market: {label}<br>"
                + f"end_date_iso: {end_date_iso}<br>"
                + f"condition_id: {condition_id}<br>"
                + "%{x|%Y-%m-%d %H:%M}<br>"
                + "Signed token position: %{y:,.2f}<extra></extra>"
            ),
        ),
        row=3,
        col=1,
    )

for row in (1, 2, 3):
    fig_token_pos.add_hline(y=0, line_dash="dash", line_color="grey", line_width=1, opacity=0.5, row=row, col=1)

fig_token_pos.update_yaxes(title_text="Tokens", row=1, col=1)
fig_token_pos.update_yaxes(title_text="Tokens", row=2, col=1)
fig_token_pos.update_yaxes(title_text="Tokens", row=3, col=1)
fig_token_pos.update_xaxes(title_text="Time", row=3, col=1)
fig_token_pos.update_layout(
    template="plotly_white",
    height=1200,
    title="Stage 2 copy-trade backtest - signed token positions",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
)
fig_token_pos.show(renderer="browser")


In [54]:
fig_usdc_pos = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.05,
    subplot_titles=(
        "Total signed USDC position (using traded price)",
        f"Per-wallet signed USDC position (top {TOP_WALLETS_POSITION} by max abs USDC position)",
        f"Per-market signed USDC position (top {TOP_MARKETS_POSITION} by max abs USDC position)",
    ),
)

fig_usdc_pos.add_trace(
    go.Scatter(
        x=usdc_total_ts["dt"],
        y=usdc_total_ts["cum_usdc_position"],
        mode="lines",
        line_shape="hv",
        name="Total USDC position",
        line=dict(width=3, color="#d62728"),
        hovertemplate="%{x|%Y-%m-%d %H:%M}<br>Signed USDC position: %{y:,.2f}<extra></extra>",
    ),
    row=1,
    col=1,
)

for wallet in top_wallets_position["wallet"]:
    sub = wallet_position_ts[wallet_position_ts["wallet"] == wallet]
    label = sub["wallet_label"].iloc[0]
    max_abs = sub["cum_usdc_position"].abs().max()
    fig_usdc_pos.add_trace(
        go.Scatter(
            x=sub["dt"],
            y=sub["cum_usdc_position"],
            mode="lines",
            line_shape="hv",
            name=f"{label} ({max_abs:,.1f})",
            hovertemplate=(
                f"Wallet: {wallet}<br>"
                + "%{x|%Y-%m-%d %H:%M}<br>"
                + "Signed USDC position: %{y:,.2f}<extra></extra>"
            ),
        ),
        row=2,
        col=1,
    )

for condition_id in top_markets_position["condition_id"]:
    sub = market_position_ts[market_position_ts["condition_id"] == condition_id]
    label = sub["market_label"].iloc[0]
    end_date_iso = sub["end_date_iso"].iloc[0]
    max_abs = sub["cum_usdc_position"].abs().max()
    fig_usdc_pos.add_trace(
        go.Scatter(
            x=sub["dt"],
            y=sub["cum_usdc_position"],
            mode="lines",
            line_shape="hv",
            name=f"{label} ({max_abs:,.1f})",
            hovertemplate=(
                f"Market: {label}<br>"
                + f"end_date_iso: {end_date_iso}<br>"
                + f"condition_id: {condition_id}<br>"
                + "%{x|%Y-%m-%d %H:%M}<br>"
                + "Signed USDC position: %{y:,.2f}<extra></extra>"
            ),
        ),
        row=3,
        col=1,
    )

for row in (1, 2, 3):
    fig_usdc_pos.add_hline(y=0, line_dash="dash", line_color="grey", line_width=1, opacity=0.5, row=row, col=1)

fig_usdc_pos.update_yaxes(title_text="USDC", row=1, col=1)
fig_usdc_pos.update_yaxes(title_text="USDC", row=2, col=1)
fig_usdc_pos.update_yaxes(title_text="USDC", row=3, col=1)
fig_usdc_pos.update_xaxes(title_text="Time", row=3, col=1)
fig_usdc_pos.update_layout(
    template="plotly_white",
    height=1200,
    title="Stage 2 copy-trade backtest - signed USDC positions",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
)
fig_usdc_pos.show(renderer="browser")
